In [1]:
# Nobite Analytics
## Goal Understand the structure and quality of the "raw" Sales, Inventory and PLZ datasets
#monday 7.09.2026:

In [16]:
import pandas as pd
import numpy as np
import os
import re

In [17]:
#first step looking at umsatzstatistik jan 2026

df_plz = pd.read_excel("data/reference/Liste-der-PLZ-in-Excel-Karte-Deutschland-Postleitzahlen.xlsx")

plz_clean = df_plz.copy()
plz_clean['PLZ'] = plz_clean['PLZ'].astype(str).str.zfill(5)

In [18]:
df_sales = pd.read_excel("data/raw/sales_2026/Umsatzstatistik Tropical Concept 31.01.2026.xlsx")


In [19]:
### PLZ Cleaning sales & plz
def clean_text(text):
    return text.replace('¸', 'ü').replace('ˆ', 'ö')

In [65]:
#cleaning function for sales
def clean_sales(df, plz_df):
    df = df.copy()

    df['PLZ'] = df['PLZ'].astype(str).str.zfill(5)
    df['BestellNr.'] = df['BestellNr.'].astype(str)
    df['Artikel'] = df['Artikel'].astype(str)
    df['PZN'] = df['PZN'].astype(str)

    df['Ort'] = df['Ort'].apply(clean_text)
    df['Bezeichnung'] = df['Bezeichnung'].apply(clean_text)

    df['Umsatz'] = df['Umsatz'].apply(
        lambda x: x.replace(',', '.') if isinstance(x, str) else x
    )

    df['Umsatz'] = pd.to_numeric(
        df['Umsatz'],
        errors='coerce'
    )

    free_delivery_mask = (
        df['Umsatz'].isna()
        & df['Auftragsart'].eq('Kostenlose Lieferung')
    )

    df.loc[free_delivery_mask, 'Umsatz'] = 0

    df['Versanddatum'] = pd.to_datetime(
        df['Versanddatum'],
        dayfirst=True,
        errors='coerce'
    )

    df['Berichtsmonat'] = df['Versanddatum'].dt.to_period('M').astype(str)

    df = df.merge(plz_df, on='PLZ', how='left')

    return df

In [67]:
sales_files = [
    "data/raw/sales_2026/" + f
    for f in os.listdir("data/raw/sales_2026")
    if f.endswith(".xlsx")
]

data_frames = []

for file in sales_files:
    df = pd.read_excel(file)
    df = clean_sales(df, plz_clean)
    data_frames.append(df)

sales_all = pd.concat(data_frames, ignore_index=True)

In [68]:
sales_all['Umsatz'].dtype

dtype('float64')

In [69]:
sales_all['Umsatz'].isna().sum()

0

In [70]:
sales_test.head()

,Versanddatum,Auftragsart,Qualifier,Beschreibung,Auftrag,Auftraggeber,BestellNr.,Kundengrupp,Name,Strasse,...,PZN,Bezeichnung,Liefermenge,Umsatz,Empfangszeit,Sendung,Berichtsmonat,Bundesland,Kreis,Typ
0,2026-01-02,Normalauftrag,Auftragsposition,Verkauf,615752,2158,3383,Pharma Pri,Richard Kehr GmbH & Co. KG,Sudetenstraﬂe 8,...,7338535,NOBITE Hautspray 100 ml,36,299.52,2025-12-30 12:30:28,9614926,2026-01,Niedersachsen,Braunschweig,Stadt
1,2026-01-05,Normalauftrag,Auftragsposition,Verkauf,615885,568211765,56141,ANZAG,Alliance Healthcare Deutschland GmbH Niederlas...,Plauener Str. 161,...,325707,NOBITE Kleidung Spray 100 ml,24,199.68,2026-01-02 10:26:34,9616810,2026-01,Berlin,Berlin,Stadt
2,2026-01-05,Normalauftrag,Auftragsposition,Verkauf,615982,2180,91536,PHOENIX,PHOENIX Pharmahandel GmbH & Co. KG Vertriebsze...,Am Kindleber Feld 3,...,7338535,NOBITE Hautspray 100 ml,12,99.84,2026-01-02 12:02:30,9617516,2026-01,Thüringen,Gotha,Kreis
3,2026-01-05,Normalauftrag,Auftragsposition,Verkauf,615969,2155,81548,PHOENIX,PHOENIX Pharmahandel GmbH & Co. KG Vertriebsze...,Lengeder Straﬂe 42,...,325707,NOBITE Kleidung Spray 100 ml,12,99.84,2026-01-02 12:01:52,9617495,2026-01,Berlin,Berlin,Stadt
4,2026-01-05,Normalauftrag,Auftragsposition,Verkauf,615862,2227,49965,PHOENIX,PHOENIX Pharmahandel GmbH & Co. KG Vertriebsze...,Schurwaldstraﬂe 14-16,...,325707,NOBITE Kleidung Spray 100 ml,12,99.84,2026-01-02 10:12:14,9616702,2026-01,Baden-Württemberg,Esslingen,Kreis


In [71]:
sales_test = clean_sales(df_sales, plz_clean)

In [72]:
##cleaning function inevntory

In [73]:
def clean_inventory(df):
    df = df.copy()

    df['ArtikelNr'] = df['ArtikelNr'].astype(str)
    df['Charge'] = df['Charge'].astype(str)
    df['Bezeichnung'] = df['Bezeichnung'].apply(clean_text)

    df[['Zugang', 'Abgang', 'Retouren', 'sonstige Vernichtung', 'Inventuren']] = (
        df[['Zugang', 'Abgang', 'Retouren', 'sonstige Vernichtung', 'Inventuren']].fillna(0)
    )

    return df

In [74]:
inventory_test.shape

(25, 22)

In [75]:
inventory_test.head()

,PZN,ArtikelNr,Bezeichnung,WEDatum,Charge,Verfalldatum,Status,Anfangsbestand,Zugang,Abgang,...,sonstige Vernichtung,Inventuren,Vernichtung R¸-Mu,Bruch SK,Vernichtung Retouren,Verdeckter Bruch WE,R¸-Mu,Verdeckter Bruch,Sonstige,Endbestand
0,NaN,325707,NOBITE Kleidung Spray 100 ml,2021-06-22,21013233,2025-09-30,4,2,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
1,NaN,325707,NOBITE Kleidung Spray 100 ml,2023-03-10,22017398,2025-10-31,4,14,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
2,NaN,325707,NOBITE Kleidung Spray 100 ml,2024-07-18,24020137,2029-06-30,4,1,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,NaN,325707,NOBITE Kleidung Spray 100 ml,2025-02-19,25020897,2028-01-31,2,3258,3360.0,-2647.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3971
4,NaN,4259189,NOBITE Verdünner Flasche 100ml,2023-08-04,22017249,2027-08-31,4,3,0.0,0.0,...,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3


In [76]:
df_inventory = pd.read_excel(
    "data/raw/inventory_2026/LagerlisteTropical-2026-01-31.xlsx"
)

In [77]:
inventory_test = clean_inventory(df_inventory)

In [78]:
# Load, clean and combine all sales files
df['Versanddatum'] = pd.to_datetime(
    df['Versanddatum'],
    dayfirst=True,
    errors='coerce'
)

df['Berichtsmonat'] = df['Versanddatum'].dt.to_period('M').astype(str)

sales_files = [
    "data/raw/sales_2026/" + f
    for f in os.listdir("data/raw/sales_2026")
    if f.endswith(".xlsx")
]

data_frames = []

for file in sales_files:
    df = pd.read_excel(file)
    df = clean_sales(df, plz_clean)
    data_frames.append(df)

sales_all = pd.concat(data_frames, ignore_index=True)

In [79]:
sales_all.shape

(3490, 23)

In [167]:
# Load, clean and combine Sales files from 2024, 2025 and 2026

sales_folders = [
    "data/raw/sales_2024",
    "data/raw/sales_2025",
    "data/raw/sales_2026"
]

sales_files = []

for folder in sales_folders:
    for file in os.listdir(folder):
        if file.endswith((".xlsx", ".xls", ".xlsm")):
            sales_files.append(os.path.join(folder, file))

data_frames = []

for file in sales_files:
    df = pd.read_excel(file)
    df = clean_sales(df, plz_clean)
    data_frames.append(df)

sales_all = pd.concat(data_frames, ignore_index=True)

In [168]:
sales_all['Berichtsmonat'].unique()

array(['2024-10', '2024-04', '2024-07', '2024-01', '2024-09', '2024-02',
       '2024-06', '2024-05', '2024-11', '2024-12', '2024-08', '2024-03',
       '2025-10', '2025-04', '2025-09', '2025-05', '2025-08', '2025-03',
       '2025-11', '2025-02', '2025-12', '2025-06', '2025-07', '2025-01',
       '2026-01', '2026-07', '2026-06', '2026-02', '2026-03', '2026-08',
       '2026-05', '2026-04'], dtype=object)

In [169]:
sales_all['Berichtsmonat'].dtype

dtype('O')

In [170]:
periods = sorted(sales_all['Berichtsmonat'].unique())

periods

['2024-01',
 '2024-02',
 '2024-03',
 '2024-04',
 '2024-05',
 '2024-06',
 '2024-07',
 '2024-08',
 '2024-09',
 '2024-10',
 '2024-11',
 '2024-12',
 '2025-01',
 '2025-02',
 '2025-03',
 '2025-04',
 '2025-05',
 '2025-06',
 '2025-07',
 '2025-08',
 '2025-09',
 '2025-10',
 '2025-11',
 '2025-12',
 '2026-01',
 '2026-02',
 '2026-03',
 '2026-04',
 '2026-05',
 '2026-06',
 '2026-07',
 '2026-08']

In [171]:
##compare the dates
current_period = periods[-1]
comparison_period = periods[-2]

current_period, comparison_period

('2026-08', '2026-07')

In [172]:
#filter the files
current_sales = sales_all[
    sales_all['Berichtsmonat'] == current_period
]

comparison_sales = sales_all[
    sales_all['Berichtsmonat'] == comparison_period
]

In [173]:
current_sales.shape, comparison_sales.shape

((399, 23), (510, 23))

In [174]:
# 1.KPI total revenue + comparison to preperiode
def total_revenue(df):
    return df['Umsatz'].sum()

In [175]:
current_revenue = total_revenue(current_sales)
comparison_revenue = total_revenue(comparison_sales)

current_revenue, comparison_revenue

(269041.67000000004, 392138.26)

In [176]:
#KPI LOGIC
#Umsatz in der aktuellen Periode ist um ca. 31,39 % niedriger als in der Vergleichsperiode
def calculate_change(current_value, comparison_value):
    if comparison_value == 0:
        return None

    return (
        (current_value - comparison_value)
        / comparison_value
        * 100
    )

In [177]:
# Test percentage change for revenue

revenue_change = calculate_change(
    current_revenue,
    comparison_revenue
)

revenue_change

-31.39111955053811

In [178]:
# Check datatype before calculating Units Sold

sales_all['Liefermenge'].dtype

dtype('int64')

In [179]:
################ KPI 2: Units Sold ##################

In [180]:
def total_units(df):
    return df['Liefermenge'].sum()

In [181]:
# Calculate current and comparison units

current_units = total_units(current_sales)
comparison_units = total_units(comparison_sales)

current_units, comparison_units

(29983, 44520)

In [182]:
# Calculate percentage change for Units Sold

units_change = calculate_change(
    current_units,
    comparison_units
)

units_change

-32.65274034141959

In [183]:
#### KPI 3 : TOP PRODUCT BY REVENUE ######

In [184]:
# Find the product with the highest revenue

def top_product(df):
    product_revenue = (
        df.groupby('Bezeichnung')['Umsatz']
        .sum()
        .sort_values(ascending=False)
    )

    return product_revenue.index[0], product_revenue.iloc[0]

In [185]:
# Find top product for current period

current_top_product, current_top_product_revenue = top_product(current_sales)

current_top_product, current_top_product_revenue

('NOBITE Kleidung Sprühflasche 200 ml', 109013.15000000001)

In [186]:
# Find top product for comparison period

comparison_top_product, comparison_top_product_revenue = top_product(comparison_sales)

comparison_top_product, comparison_top_product_revenue

('NOBITE Hautspray 100 ml', 192192.0)

In [187]:
#### KPI 4 : REVENUE CHANGE #####

In [188]:
# Revenue change between current and comparison period

revenue_change_kpi = calculate_change(
    current_revenue,
    comparison_revenue
)

revenue_change_kpi

-31.39111955053811

In [189]:
# Round Revenue Change for dashboard display

revenue_change_kpi = round(revenue_change_kpi, 2)

revenue_change_kpi

-31.39

In [190]:
################################ Kunden & Regionale Performance ##############################################

In [191]:
# KPI 1 : TOP CUSTOMER BY REVENUE 

In [192]:
# Check customer groups

sales_all['Kundengrupp'].value_counts().head(20)

Kundengrupp
ANZAG         3897
PHOENIX       3421
Noweda        2576
Sanacorp      2490
Pharma Pri     916
TC Privat      787
AEP             95
VA 002           6
JeCo             6
Apo              5
KVA              3
GH               2
Name: count, dtype: int64

In [193]:
# finding customer with the highest revenue

def top_customer(df):
    customer_revenue = (
        df.groupby('Kundengrupp')['Umsatz']
        .sum()
        .sort_values(ascending=False)
    )

    return customer_revenue.index[0], customer_revenue.iloc[0]

In [194]:
#TOP customer of the current month 
comparison_top_customer, comparison_top_customer_revenue = top_customer(comparison_sales)

comparison_top_customer, comparison_top_customer_revenue

('ANZAG', 183596.5)

In [195]:
# KPI 2 : TOP REGION IN GERMANY 

In [196]:
# Find region with the highest revenue

def top_region(df):
    region_revenue = (
        df.groupby('Bundesland')['Umsatz']
        .sum()
        .sort_values(ascending=False)
    )

    return region_revenue.index[0], region_revenue.iloc[0]

In [197]:
# Top region for current month 

current_top_region, current_top_region_revenue = top_region(current_sales)

current_top_region, current_top_region_revenue

('Nordrhein-Westfalen', 130873.02)

In [198]:
# Top region for comparison

comparison_top_region, comparison_top_region_revenue = top_region(comparison_sales)

comparison_top_region, comparison_top_region_revenue

('Nordrhein-Westfalen', 140518.58)

In [199]:
# KPI 3 : Which wholesaler made the most revenue 

In [200]:

# Calculate revenue share of the top customer

def top_customer_share(df):
    customer_revenue = (
        df.groupby('Kundengrupp')['Umsatz']
        .sum()
        .sort_values(ascending=False)
    )

    top_customer_revenue = customer_revenue.iloc[0]
    total_revenue_value = df['Umsatz'].sum()

    return (
        top_customer_revenue
        / total_revenue_value
        * 100
    )

In [201]:
# Top customer share for current month 

current_top_customer_share = top_customer_share(current_sales)

current_top_customer_share

57.436195664411386

In [202]:
# Top customer share for comparison

comparison_top_customer_share = top_customer_share(comparison_sales)

comparison_top_customer_share

46.81932846848456

In [203]:
# Change in top customer share

top_customer_share_change = calculate_change(
    current_top_customer_share,
    comparison_top_customer_share
)

top_customer_share_change

22.67624834276158

In [204]:
# KPI 4 : REVENUE difference  

In [205]:
# Find the same month in the previous year

current_period_date = pd.Period(current_period, freq='M')
previous_year_period = str(current_period_date - 12)

current_period, previous_year_period

('2026-08', '2025-08')

In [210]:
previous_year_period in periods

True

In [212]:
# Filter sales data for the same month in the month year

previous_year_sales = sales_all[
    sales_all['Berichtsmonat'] == previous_year_period
]

previous_year_sales.shape

(599, 23)

In [214]:
# Calculate revenue for the previous-year month

previous_year_revenue = total_revenue(previous_year_sales)

current_revenue, previous_year_revenue

(269041.67000000004, 376830.96)

In [215]:
# Calculate revenue change for year vs year

revenue_change_yoy = calculate_change(
    current_revenue,
    previous_year_revenue
)

revenue_change_yoy

-28.604149191987826